Aggregates points of a dataset in a similar area into a new aggregated data point

# Initialization

In [ ]:
# Imports
from typing import Callable, List, Optional, Tuple
import os
import pandas as pd

# my scripts
from pain2map import AggregationManager, AggregatedPainData, PainData, Coordinate

In [ ]:
# Constants
BASE_PATH = os.path.join('..', 'data', 'actual')
DATA_FILE = os.path.join(BASE_PATH, 'normalized_ctr.csv')   # TODO: use full dataset
OUT_FILE = os.path.join(BASE_PATH, 'aggr_ctr.csv')
OUT_FILE2 = os.path.join(BASE_PATH, 'aggr_ctr2.csv')
OUT_FILE3 = os.path.join(BASE_PATH, 'aggr_ctr3.csv')
OUT_FILE4 = os.path.join(BASE_PATH, 'aggr_ctr4.csv')

In [ ]:
# Helper Functions
def aggrdp_to_df(aggr_data: List[AggregatedPainData], origin: str) -> pd.DataFrame:
  data = []
  for data_point in aggr_data:
    data.append({
        'lat': data_point.lat,
        'lon': data_point.lng,
        'value': data_point.val,
        'datatype': 'CO2_Emissions',
        'painorigin': origin
    })
  return pd.DataFrame(data)

In [ ]:
dataset = pd.read_csv(DATA_FILE)

aggr_man = AggregationManager(36, 18)
aggr_data = aggr_man.aggregate(dataset, coor_func=AggregatedPainData.mid_point_coordinate)

In [ ]:
#df_aggr = aggrdp_to_df(aggr_data, origin="Aggr_area-center")
#df_aggr.to_csv(OUT_FILE, index=True, index_label='id')

# Visualizing

In [ ]:
# Imports
import matplotlib.pyplot as plt

In [ ]:
depths = [dp.depth for dp in aggr_data]
plt.plot(depths)

In [ ]:
sort_depths = list(depths)
sort_depths.sort()
plt.plot(sort_depths)

# AggrConfig

In [ ]:
class AggrConfig:
  @staticmethod
  def coor_func_from_id(center_func_id: int) -> Tuple[Callable[[List[PainData], Coordinate], Coordinate], str]:
    if center_func_id == 0:
      return AggregatedPainData.center_coordinate, "area"
    elif center_func_id == 1:
      return AggregatedPainData.mid_point_coordinate, "data-mid"
    elif center_func_id == 2:
      return AggregatedPainData.max_point_coordinate, "data-max"
    elif center_func_id == 3:
      return AggregatedPainData.weighted_mid_point_coordinate, "weighted-mid"
    else:
      raise Exception(f"Illegal center_func_id={center_func_id}")

  def __init__(self, cols: int, rows: int, center_func: int, label: Optional[str] = None):
    self.cols = cols
    self.rows = rows
    self.center_func = center_func
    self.label = label

  @property
  def origin(self) -> str:
    return "Aggr_" + (self.label if self.label else \
      f"{self.rows}x{self.cols} {AggrConfig.coor_func_from_id(self.center_func)[1]}-centric")
  
  def get_aggr_data(self, dataset: pd.DataFrame) -> List[AggregatedPainData]:
    aggr_man = AggregationManager(self.cols, self.rows)
    return aggr_man.aggregate(dataset, coor_func=AggrConfig.coor_func_from_id(self.center_func)[0])
  
  def aggregate(self, dataset: pd.DataFrame) -> pd.DataFrame:
    aggr_data = self.get_aggr_data(dataset)
    return aggrdp_to_df(aggr_data, origin=self.origin)
  
  def get_file_path(self) -> str:
    return os.path.join(BASE_PATH, f"aggr_ctr_{self.origin}.csv")


# 36x18 Aggregation Layers

In [ ]:
configs: List[AggrConfig] = [
  AggrConfig(36, 18, 0),
  AggrConfig(36, 18, 1),
  AggrConfig(36, 18, 2),
  AggrConfig(36, 18, 3),
]

dataset = pd.read_csv(DATA_FILE)
for config in configs:
  df_aggr = config.aggregate(dataset)
  df_aggr.to_csv(config.get_file_path(), index=True, index_label='id')

In [ ]:
# combine different configs (and regular layer data) into one csv for the database to load
dfs = []
for config in configs:
  dfs.append(pd.read_csv(config.get_file_path(), index_col=False))

# also add regular layer data
if True:
  df_socioeco = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_01.csv"), index_col=False)
  df_phys = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_05.csv"), index_col=False)
  df_emo = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_10.csv"), index_col=False)
  df_env = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_50.csv"), index_col=False)
  dfs += [df_socioeco, df_phys, df_emo, df_env]

dataset = pd.concat(
    [df.drop(columns=["id"]) for df in dfs],
    ignore_index=True
)
dataset.to_csv(OUT_FILE, index=True, index_label="id")

# 72x36 Aggregation Layers

In [ ]:
db_dataset = pd.read_csv(OUT_FILE).drop(columns=["id"])
db_dataset.head()

In [ ]:
configs: List[AggrConfig] = [
  AggrConfig(72, 36, center_func=0),
  AggrConfig(72, 36, center_func=1),
  AggrConfig(72, 36, center_func=2),
  AggrConfig(72, 36, center_func=3),
]

dataset = pd.read_csv(DATA_FILE)
for config in configs:
  df_aggr = config.aggregate(dataset)
  db_dataset = pd.concat([db_dataset, df_aggr], ignore_index=True)
db_dataset.to_csv(OUT_FILE2, index_label="id")

# Some detailed Data Points

In [ ]:
db_dataset = pd.read_csv(OUT_FILE2).drop(columns=["id"])
db_dataset.head()

In [ ]:
# 
FILTER_STEP = 16
configs: List[AggrConfig] = [
  AggrConfig(36, 18, center_func=0, label="36x18_details"),
  AggrConfig(72, 36, center_func=0, label="72x36_details"),
]

dataset = pd.read_csv(DATA_FILE)
for config in configs:
  aggr_points = config.get_aggr_data(dataset)
  data_points: PainData = []
  # get the detailed points of every #FILTER_STEP aggregated point
  for i in range(0, len(aggr_points), FILTER_STEP):
    data_points += aggr_points[i].resolve()
  # concatenate
  df_details = pd.DataFrame([dp.to_df_object(config.origin) for dp in data_points])
  db_dataset = pd.concat([db_dataset, df_details], ignore_index=True)
db_dataset.to_csv(OUT_FILE3, index_label="id")

# 360x180 Aggregation Layers

In [ ]:
db_dataset = pd.read_csv(OUT_FILE3).drop(columns=["id"])
db_dataset.head()

In [ ]:
configs: List[AggrConfig] = [
  AggrConfig(360, 180, center_func=0),
  AggrConfig(360, 180, center_func=1),
  AggrConfig(360, 180, center_func=2),
  AggrConfig(360, 180, center_func=3),
]

dataset = pd.read_csv(DATA_FILE)
for config in configs:
  df_aggr = config.aggregate(dataset)
  db_dataset = pd.concat([db_dataset, df_aggr], ignore_index=True)
db_dataset.to_csv(OUT_FILE4, index_label="id")